In [219]:
import pandas as pd
from keras.src.metrics import accuracy
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [220]:
df = pd.read_csv("./datasets/citrus.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      10000 non-null  object 
 1   diameter  10000 non-null  float64
 2   weight    10000 non-null  float64
 3   red       10000 non-null  int64  
 4   green     10000 non-null  int64  
 5   blue      10000 non-null  int64  
dtypes: float64(2), int64(3), object(1)
memory usage: 468.9+ KB


In [221]:
df.head()

,name,diameter,weight,red,green,blue
0,orange,2.96,86.76,172,85,2
1,orange,3.91,88.05,166,78,3
2,orange,4.42,95.17,156,81,2
3,orange,4.47,95.60,163,81,4
4,orange,4.48,95.76,161,72,9


### Transform category column to numerik

In [222]:
names_pure = df["name"].unique()
names_transform = {value: key for key, value in enumerate(names_pure)}

In [223]:
# WARNING DON'T RERUN MORE THAN ONCE TIME
# df["name"] = df["name"].replace(names_transform) # Deprecated
df["name"] = df["name"].map(names_transform)
df.head()

,name,diameter,weight,red,green,blue
0,0,2.96,86.76,172,85,2
1,0,3.91,88.05,166,78,3
2,0,4.42,95.17,156,81,2
3,0,4.47,95.60,163,81,4
4,0,4.48,95.76,161,72,9


In [224]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      10000 non-null  int64  
 1   diameter  10000 non-null  float64
 2   weight    10000 non-null  float64
 3   red       10000 non-null  int64  
 4   green     10000 non-null  int64  
 5   blue      10000 non-null  int64  
dtypes: float64(2), int64(4)
memory usage: 468.9 KB


### Split datasets range X = 1-5 y = 5

In [225]:
datasets = df.values

In [226]:
X = datasets[:,1:6] # Take first column up to 5
y = datasets[:,0] # Take sixth column

### Scaler training data (X)

In [227]:
scaler = MinMaxScaler()

X_scaler = scaler.fit_transform(X)
X_scaler

array([[0.        , 0.        , 0.74025974, 0.63529412, 0.        ],
       [0.07042254, 0.00738197, 0.66233766, 0.55294118, 0.01851852],
       [0.10822832, 0.04812589, 0.53246753, 0.58823529, 0.        ],
       ...,
       [0.93624907, 0.97133047, 0.68831169, 0.6       , 0.33333333],
       [0.96071164, 0.99216023, 0.35064935, 0.48235294, 0.16666667],
       [1.        , 1.        , 0.48051948, 0.50588235, 0.        ]])

### Split to train and test data

In [228]:
X_train, X_test, y_train, y_test = train_test_split(X_scaler, y, train_size=0.3, random_state=42)

### Make model and compile

In [229]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential([
    Dense(32, activation="relu", input_shape=(5,)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="sgd", loss="binary_crossentropy", metrics=["accuracy"])

In [230]:
model.fit(X_train, y_train, epochs=200)

Epoch 1/200
94/94 [==============================] - 3s 5ms/step - loss: 0.6493 - accuracy: 0.5883
Epoch 2/200
94/94 [==============================] - 0s 4ms/step - loss: 0.6195 - accuracy: 0.7823
Epoch 3/200
94/94 [==============================] - 0s 4ms/step - loss: 0.5832 - accuracy: 0.8567
Epoch 4/200
94/94 [==============================] - 0s 4ms/step - loss: 0.5360 - accuracy: 0.8823
Epoch 5/200
94/94 [==============================] - 0s 4ms/step - loss: 0.4817 - accuracy: 0.8993
Epoch 6/200
94/94 [==============================] - 0s 4ms/step - loss: 0.4231 - accuracy: 0.9117
Epoch 7/200
94/94 [==============================] - 0s 4ms/step - loss: 0.3657 - accuracy: 0.9167
Epoch 8/200
94/94 [==============================] - 0s 4ms/step - loss: 0.3159 - accuracy: 0.9183
Epoch 9/200
94/94 [==============================] - 0s 4ms/step - loss: 0.2769 - accuracy: 0.9230
Epoch 10/200
94/94 [==============================] - 0s 4ms/step - loss: 0.2491 - accuracy: 0.9253
Epoch 11/

### Evaluate model

In [231]:
model.evaluate(X_test, y_test, batch_size=1)

7000/7000 [==============================] - 24s 3ms/step - loss: 0.1789 - accuracy: 0.9289


[0.1789085865020752, 0.928857147693634]

In [232]:
one_of_data_train = df.loc[9998].drop("name").values.reshape(1, -1)

pred = model.predict([one_of_data_train])
result = int(pred[0][0])

names_reversed = dict(map(reversed, names_transform.items()))
print(names_reversed[result])

1/1 [==============================] - 0s 182ms/step
grapefruit


In [233]:
names_reversed

{0: 'orange', 1: 'grapefruit'}